# Experiment 2: **Ablation Study**

## Objective
In this experiment, we compute the evaluation metrics for different ablation configurations. We extend the results obtained in the original paper by computing all possible ablation configurations, analyzing their impact on performance. This is referred to in our paper as **Table 6**.

---

## Methodology
- **Models Used in Our Experiment:**  
  - GPT-4o Mini  
  - Qwen2.5-72B (int4)  

- **Evaluation Metrics:**  
  The script will compute the following metrics for each ablation configuration:  
  - **% 5/6-way agreement**  
  - **% 6-way agreement**  
  - **% Any agreement**  

---

## Results
The results of this experiment will provide insights into the impact of different ablation configurations on model performance. These findings are presented in **Table 6** of our paper.

In [2]:
import os
import eval_utils as evaluation
import json
import numpy as np
import pandas as pd
from IPython.display import display

raw_path = '../our_games_descriptions/base/output/changing_ablation'

models = ['gpt4o-mini', 'Qwen2.5-72B-Instruct']
ablation_configs = [
    "ablation_0000",
    "ablation_0001",
    "ablation_0010",
    "ablation_0011",
    "ablation_0100",
    "ablation_0101",
    "ablation_0110",
    "ablation_0111",
    "ablation_1000",
    "ablation_1001",
    "ablation_1010",
    "ablation_1011",
    "ablation_1100",
    "ablation_1101",
    "ablation_1110",
    "ablation_1111"
]

results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6

for model in models:

    results[model] = {}

    for ablation_config in ablation_configs:
        
        directory = os.path.join(raw_path, model, ablation_config)
        agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
        answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

        num_rounds = 0
        for file_ in answers_files:
            answers = json.load(open(file_))
            _num_rounds = len(answers['rounds'])
            num_rounds = max(num_rounds, _num_rounds)

        # Track statistics
        feasible_in_last_step = 0
        accepted_by_all_in_last_step = 0
        contained_feasible_deal = 0
        successfull_games = 0
        total_rounds = 0

        # Loop through all answer files (each represents a game)
        for file_ in answers_files:
            answers = json.load(open(file_))
            
            if len(answers['rounds']) != num_rounds:
                print(f"WARNING: Game {file_} has a different number of rounds")
                continue
            total_rounds += len(answers['rounds'])
            successfull_games += 1

            # Extract deals for this game
            feasible_found = False

            # Extract the name of the first player (p1) to validate feasibility throughout the game
            p1_name = answers['rounds'][0]['agent']

            total_deals = 0
            
            for i, round_ in enumerate(answers['rounds']):
                name, answer = round_['agent'], round_['public_answer']
                deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

                try:
                    deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
                except:
                    print(f"Error in game {file_} round {i}")
                    continue

                if issues_suggested >= ISSUES_NUM:
                    total_deals += 1

                # Check if the deal was feasible at any point (Deal must have been proposed by p1)
                if evaluation.is_feasible(agents, deal) and name == p1_name:
                    feasible_found = True
            

            # CHECK GAME COMPLETION METRICS

            last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
            
            # 1. Check if the last deal is feasible
            if evaluation.is_feasible(agents, last_deal):
                feasible_in_last_step += 1

            # 2. Check if the last deal is acceptable by all agents
            all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
            if all_accept:
                accepted_by_all_in_last_step += 1

            # 3. Check if any deal during the game was in the feasibility set
            if feasible_found:
                contained_feasible_deal += 1

        # Compute percentages
        num_games = successfull_games
        perc_feasible_last = (feasible_in_last_step / num_games) * 100
        perc_accepted_all_last = (accepted_by_all_in_last_step / num_games) * 100
        perc_feasible_any = (contained_feasible_deal / num_games) * 100

        results[model][ablation_config] = {
            "5/6-way (%)": f"{perc_feasible_last:.2f}",
            "6-way (%)": f"{perc_accepted_all_last:.2f}",
            "Any (%)": f"{perc_feasible_any:.2f}",
        }


# Print results
df = pd.DataFrame(results)

# Define symbol mapping for ablation (0 → ✓, 1 → ✗)
def ablation_to_symbols(ablation_str):
    return [ "✓" if bit == "0" else "✗" for bit in ablation_str.split("_")[-1] ]

# Initialize table storage
table_data = []

# Extract models
models = list(results.keys())

# Iterate over all ablation configs
for ablation in ablation_configs:
    row = ablation_to_symbols(ablation)  # Convert to ✓ / ✗ symbols
    
    # Store metrics for both models in "model1 / model2" format
    metrics = []
    for metric in ["5/6-way (%)", "6-way (%)", "Any (%)"]:
        val1 = results[models[0]][ablation][metric]
        val2 = results[models[1]][ablation][metric]
        metrics.append(f"{val1} / {val2}")
    
    table_data.append(row + metrics)

# Define column names
columns = ["Prev. deals", "Others. Prefer.", "Candidates", "Planning", "5/6-way", "6-way", "Any"]

# Convert to DataFrame
df = pd.DataFrame(table_data, columns=columns)

# Display nicely formatted table
df_styled = df.style.set_properties(**{"text-align": "center"}) \
                    .set_caption("Ablation Study Results") \
                    .set_table_styles([
                        {'selector': 'th', 'props': [('font-size', '14px'), ('text-align', 'center')]},
                        {'selector': 'td', 'props': [('font-size', '13px')]}
                    ])

display(df_styled)



,Prev. deals,Others. Prefer.,Candidates,Planning,5/6-way,6-way,Any
0,✓,✓,✓,✓,60.00 / 90.00,0.00 / 0.00,100.00 / 100.00
1,✓,✓,✓,✗,55.00 / 95.00,10.00 / 0.00,95.00 / 95.00
2,✓,✓,✗,✓,60.00 / 90.00,0.00 / 0.00,85.00 / 90.00
3,✓,✓,✗,✗,45.00 / 80.00,10.00 / 0.00,95.00 / 100.00
4,✓,✗,✓,✓,5.00 / 90.00,0.00 / 0.00,10.00 / 90.00
5,✓,✗,✓,✗,35.00 / 85.00,10.00 / 5.00,80.00 / 100.00
6,✓,✗,✗,✓,30.00 / 90.00,10.00 / 0.00,85.00 / 95.00
7,✓,✗,✗,✗,55.00 / 85.00,15.00 / 0.00,85.00 / 100.00
8,✗,✓,✓,✓,60.00 / 85.00,10.00 / 0.00,90.00 / 95.00
9,✗,✓,✓,✗,60.00 / 80.00,15.00 / 5.00,95.00 / 90.00
